IMPORTING OF IMPORTANT MODULES

In [5]:
def import_modules():
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import os
    import seaborn as sb
    import openpyxl
    from pylab import rcParams
    from scipy.stats import pearsonr
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import OneHotEncoder
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    from sklearn.svm import SVR
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import KFold
    from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
    from xgboost import XGBRegressor
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.preprocessing import MinMaxScaler
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense

IMPORTING OF DATASET

In [6]:
def import_dataset():
    pwd = os.getcwd()
    dataset = pd.read_excel(pwd + "/data/DATA.xlsx")
    dataset_copy = dataset.copy()
    dataset_copy.drop(['Time','Date'], inplace=True, axis=1)
    dataset_copy.set_index('Datetime', inplace=True)

VISUALIZING DATASET

In [7]:
def visualize_dataset(Data, Title):
    plt.figure(figsize=(20, 6))
    plt.plot(Data['Power'])
    plt.xlabel('Date')
    plt.ylabel('Power')
    plt.title(Title)
    plt.axhline(y=Data['Power'].mean(), color='r', linestyle='--', label='Mean')
    plt.legend()
    plt.show()

HANDLING OUTLIET USING INTERQUARTILE RANGE 

In [8]:
upper = dataset_copy['Power'].quantile(0.75)
lower = dataset_copy['Power'].quantile(0.25)
IQR = upper - lower
max_threshold = upper + (1.5 * IQR)
min_threshold = lower - (1.5 * IQR)
dataset_copy['Power'] = np.where(dataset_copy['Power']>max_threshold, max_threshold, np.where(dataset_copy['Power']<min_threshold, min_threshold,dataset_copy['Power'] ))

MEAN IMPUTATION

In [9]:
mean = dataset_copy['Power'].mean()
dataset_copy['Power']= dataset_copy['Power'].fillna(mean)

CORRELATION OF FEATURES

In [ ]:
corr = dataset_copy.corr(numeric_only=True)
rcParams['figure.figsize'] = 14.7,8.27
sb.heatmap(corr, 
           xticklabels=corr.columns.values, 
           yticklabels=corr.columns.values, 
           cmap="YlGnBu",
          annot=True)

    TRAIN-TEST SPLIT

In [ ]:
def train_val_test_split(dataset):
    X = dataset.iloc[:,:-1].values
    Y = dataset.iloc[:,7:].values
    ct = ColumnTransformer(transformers = [('encoder', OneHotEncoder(), [6])], remainder = 'passthrough')
    X =  np.array(ct.fit_transform(X))
    X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

DATA NORMALIZATION

In [ ]:
def data_normalization(dataset):
    sc = StandardScaler()
    X_train[:,11] = sc.fit_transform(X_train[:,11].reshape(-1, 1)).flatten()
    X_test[:,11] = sc.fit_transform(X_test[:,11].reshape(-1, 1)).flatten()
    X_val[:,11] = sc.fit_transform(X_val[:,11].reshape(-1, 1)).flatten()        

MODEL INITIALIZATION 

In [ ]:
def create_SVR():
    model = SVR(kernel='rbf', gamma= 'auto',   C= 20000)  

def create_XGBoost():
    model = XGBRegressor(n_estimators=320, learning_rate=0.2, max_depth = 6,  early_stopping_rounds=50) 

def create_SVR():
    model = RandomForestRegressor(n_estimators=235, max_depth= 23, random_state=42) 



MODEL TRAINING

In [ ]:
def train_model(model, X_train, y_train):
    model.fit(X_train, y_train)

MODEL PREDICTION 

In [ ]:
def predict(model, data):
    predict = model.predict(data)

EVALUATE MODEL

In [ ]:
def evaluate(actual, predict):
    mse = mean_squared_error(actual, predict)
    mae = mean_absolute_error(actual, predict)
    mape = mean_absolute_percentage_error(actual, predict)

PLOT COMPARISON

In [ ]:
def plot_result(result)
    plt.figure(figsize=(20, 6))
    plt.plot(result['Actual'], label="Actual")
    plt.plot(result['Predicted'], label="Predicted")
    plt.xlabel('Date')
    plt.ylabel('Power')
    plt.legend()
    plt.show()


LSTM MODEL

In [ ]:
#Normalize data
scaler1 = MinMaxScaler(feature_range=(0, 1))
scaler2 = MinMaxScaler(feature_range=(0, 1))
scaled_load_data_X = scaler1.fit_transform(load_data_X)
scaled_load_data_Y = scaler2.fit_transform(load_data_Y)
scaled_load_data = np.hstack((scaled_load_data_X, scaled_load_data_Y))


# Split data into train and test sets
train_size = int(len(scaled_load_data) * 0.6)
val_size = (len(scaled_load_data) - train_size)/2
test_size = val_size
limit = int(test_size + train_size)
train_data, val_data, test_data = scaled_load_data[0:train_size, :], scaled_load_data[train_size:limit :], scaled_load_data[limit:len(scaled_load_data), :]


# Function to create dataset with look back
def create_sequences(dataset, time_steps=1):
    X, Y = [], []
    for i in range(len(dataset) - time_steps - 1):
        a = dataset[i:(i + time_steps), :]
        X.append(a)
        Y.append(dataset[i + time_steps, -1])  
    return np.array(X), np.array(Y)

look_back = 24  

# Create dataset with look back
X_train, Y_train = create_sequences(train_data, look_back)
X_val, Y_val = create_sequences(val_data, look_back)
X_test, Y_test = create_sequences(test_data, look_back)

# Define LSTM model
model = Sequential()
model.add(Input(shape=(look_back, len(features))))  # Input layer with specified shape
model.add(LSTM(units=50))
model.add(Dense(units=1))  # Output layer with 1 neuron for regression
model.compile(optimizer='adam', loss='mean_squared_error')